# 02 - Data Preparation (Realistic Distribution)

Keeps real-world patterns: mostly high availability, some mid-range, few low/empty

**Target distribution (realistic):**
- 0-50 beds: 5% (rare - crisis situations)
- 50-80 beds: 15% (moderate load)
- 80-95 beds: 30% (typical busy day)
- 95-100 beds: 50% (normal operations)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
sys.path.append(PROJECT_DIR)
!pip install -q imbalanced-learn joblib matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete!")

## Cell 2: Severity Data

In [ ]:
print("="*60)
print("PREPARING SEVERITY DATA")
print("="*60)

SEVERITY_SAMPLE_SIZE = 100000
severity_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/severity_processed.csv', nrows=SEVERITY_SAMPLE_SIZE)
print(f"Loaded: {severity_df.shape}")

target_col = 'esi'
drop_cols = ['dep_name'] if 'dep_name' in severity_df.columns else []
X = severity_df.drop(columns=[target_col] + drop_cols).astype(np.float32)
y = severity_df[target_col]

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

X = X.fillna(X.median())
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

del severity_df, X, X_train, X_val, X_test
gc.collect()

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

processed_dir = f'{PROJECT_DIR}/datasets/processed'
np.savez(f'{processed_dir}/severity_train.npz',
         X_train=X_train_balanced.astype(np.float32), X_val=X_val_scaled, X_test=X_test_scaled,
         y_train=y_train_balanced, y_val=y_val.values, y_test=y_test.values,
         feature_names=X.columns.tolist())

del X_train_balanced, X_val_scaled, X_test_scaled, y_train_balanced
gc.collect()

os.makedirs(f'{PROJECT_DIR}/models/severity', exist_ok=True)
joblib.dump(scaler, f'{PROJECT_DIR}/models/severity/scaler_severity.joblib')
print(f"✓ Severity: {len(y_train)} train → SMOTE balanced")

## Cell 3: ETA Data

In [ ]:
print("\n="*60)
print("PREPARING ETA DATA")
print("="*60)

ETA_SAMPLE_SIZE = 200000
eta_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/eta_processed.csv', nrows=ETA_SAMPLE_SIZE).astype(np.float32)
print(f"Loaded: {eta_df.shape}")

target_col = 'duration_min'
feature_cols = [c for c in eta_df.columns if c != target_col]
X = eta_df[feature_cols]
y = eta_df[target_col]

n = len(eta_df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]
y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

scaler_eta = StandardScaler()
X_train_scaled = scaler_eta.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler_eta.transform(X_val).astype(np.float32)
X_test_scaled = scaler_eta.transform(X_test).astype(np.float32)

del eta_df, X, X_train, X_val, X_test
gc.collect()

np.savez(f'{processed_dir}/eta_train.npz',
         X_train=X_train_scaled, X_val=X_val_scaled, X_test=X_test_scaled,
         y_train=y_train.values, y_val=y_val.values, y_test=y_test.values,
         feature_names=feature_cols)

del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

os.makedirs(f'{PROJECT_DIR}/models/eta', exist_ok=True)
joblib.dump(scaler_eta, f'{PROJECT_DIR}/models/eta/scaler_eta.joblib')
print(f"✓ ETA: {len(y_train)} train, {len(y_val)} val, {len(y_test)} test")

## Cell 4: Create Realistic Bed Distribution

In [ ]:
print("\n="*60)
print("CREATING REALISTIC BED DISTRIBUTION")
print("Real-world pattern: Mostly high availability, some mid/low")
print("="*60)

bed_df = pd.read_csv(f'{PROJECT_DIR}/datasets/processed/bed_availability_processed.csv')
print(f"Original: {bed_df.shape[0]} rows")

# Define realistic target distribution
TARGET_TOTAL = 10000
TARGET_DIST = {
    '0-20': 200,      # 2% - Crisis/rare
    '20-40': 300,     # 3% - Very low
    '40-60': 500,     # 5% - Low
    '60-75': 1000,    # 10% - Moderate-low
    '75-85': 1500,    # 15% - Moderate
    '85-90': 2000,    # 20% - Getting full
    '90-95': 2500,    # 25% - Nearly full
    '95-100': 3000    # 30% - Normal operations
}

print(f"\nTarget distribution (realistic):")
for bin_name, target in TARGET_DIST.items():
    pct = target / TARGET_TOTAL * 100
    print(f"  {bin_name} beds: {target} samples ({pct:.0f}%)")

# Create bins in original data
bins = [0, 20, 40, 60, 75, 85, 90, 95, 100]
labels = list(TARGET_DIST.keys())
bed_df['bed_bin'] = pd.cut(bed_df['available_beds'], bins=bins, labels=labels, include_lowest=True)

# Check what we have
original_counts = bed_df['bed_bin'].value_counts().sort_index()
print(f"\nOriginal distribution:")
print(original_counts.fillna(0))

In [ ]:
print(f"\nResampling to realistic distribution...")

realistic_dfs = []

for bin_name, target_count in TARGET_DIST.items():
    group = bed_df[bed_df['bed_bin'] == bin_name]
    actual_count = len(group)
    
    if actual_count == 0:
        # Create synthetic data for empty bins (0-20, 20-40, 40-60)
        print(f"  {bin_name}: creating {target_count} synthetic (was 0)")
        # Sample from higher bins and modify
        nearest = bed_df[bed_df['available_beds'] > 60].sample(n=min(target_count, len(bed_df[bed_df['available_beds'] > 60])), replace=True, random_state=42)
        synthetic = nearest.copy()
        # Parse bin range
        parts = bin_name.split('-')
        bin_start, bin_end = int(parts[0]), int(parts[1])
        # Assign values within range
        synthetic['available_beds'] = np.random.randint(bin_start, bin_end + 1, size=len(synthetic))
        # Add realistic noise to features (more noise = lower bed availability)
        numeric_cols = synthetic.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != 'available_beds':
                noise_factor = 0.1 if bin_end > 80 else 0.2
                noise = np.random.normal(0, synthetic[col].std() * noise_factor, len(synthetic))
                synthetic[col] = synthetic[col] + noise
        realistic_dfs.append(synthetic.head(target_count))
    
    elif actual_count > target_count:
        # Downsample
        sampled = group.sample(n=target_count, random_state=42)
        print(f"  {bin_name}: {actual_count} → {target_count} (downsampled)")
        realistic_dfs.append(sampled)
    
    elif actual_count < target_count:
        # Upsample
        upsampled = group.sample(n=target_count, replace=True, random_state=42)
        numeric_cols = upsampled.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != 'available_beds':
                noise = np.random.normal(0, upsampled[col].std() * 0.03, len(upsampled))
                upsampled[col] = upsampled[col] + noise
        print(f"  {bin_name}: {actual_count} → {target_count} (upsampled)")
        realistic_dfs.append(upsampled)
    
    else:
        print(f"  {bin_name}: {actual_count} (perfect)")
        realistic_dfs.append(group)

bed_realistic = pd.concat(realistic_dfs, ignore_index=True)
print(f"\n✓ Realistic dataset: {len(bed_realistic)} rows")

## Cell 5: Verify Realistic Distribution

In [ ]:
# Verify new distribution
bed_realistic['bed_bin_check'] = pd.cut(bed_realistic['available_beds'], bins=bins, labels=labels, include_lowest=True)
new_counts = bed_realistic['bed_bin_check'].value_counts().sort_index()

print(f"\nFinal realistic distribution:")
total = len(bed_realistic)
for bin_name in labels:
    count = new_counts.get(bin_name, 0)
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {bin_name:>6}: {count:>5} ({pct:>4.1f}%) {bar}")

print(f"\nTotal: {total} samples")
print(f"Most common: 95-100 beds (normal ops)")
print(f"Rarest: 0-20 beds (crisis - but now learnable!)")

## Cell 6: Visualize Realistic vs Original

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original (extremely imbalanced)
axes[0].hist(bed_df['available_beds'], bins=30, edgecolor='black', color='coral', alpha=0.7)
axes[0].axvline(bed_df['available_beds'].mean(), color='red', linestyle='--', label=f"Mean: {bed_df['available_beds'].mean():.1f}")
axes[0].set_xlabel('Available Beds')
axes[0].set_ylabel('Count')
axes[0].set_title(f'ORIGINAL (n={len(bed_df)})\n83% in 95-100 range - TOO BIASED!')
axes[0].legend()

# Realistic (skewed but learnable)
axes[1].hist(bed_realistic['available_beds'], bins=30, edgecolor='black', color='seagreen', alpha=0.7)
axes[1].axvline(bed_realistic['available_beds'].mean(), color='green', linestyle='--', label=f"Mean: {bed_realistic['available_beds'].mean():.1f}")
axes[1].set_xlabel('Available Beds')
axes[1].set_ylabel('Count')
axes[1].set_title(f'REALISTIC (n={len(bed_realistic)})\nSkewed but ALL ranges represented')
axes[1].legend()

plt.suptitle('Bed Availability: Original vs Realistic Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/bed_distribution_realistic.png', dpi=150)
plt.show()

print(f"\n✅ REALISTIC distribution achieved!")
print(f"   - Still mostly high availability (as in reality)")
print(f"   - But now includes rare 0-20 bed cases")
print(f"   - Model can predict ANY bed count, not just 95-100")

## Cell 7: Prepare Realistic Data for Training

In [ ]:
print("\n="*60)
print("PREPARING REALISTIC BED DATA")
print("="*60)

bed_final = bed_realistic.drop(columns=['bed_bin', 'bed_bin_check'], errors='ignore')

target_col = 'available_beds'
feature_cols = [c for c in bed_final.columns if c != target_col]

X = bed_final[feature_cols].astype(np.float32)
y = bed_final[target_col]

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

X = X.fillna(X.median())

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler_bed = StandardScaler()
X_train_scaled = scaler_bed.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler_bed.transform(X_val).astype(np.float32)
X_test_scaled = scaler_bed.transform(X_test).astype(np.float32)

del bed_df, bed_realistic, bed_final, X, X_train, X_val, X_test
gc.collect()

np.savez(f'{processed_dir}/bed_train.npz',
         X_train=X_train_scaled, X_val=X_val_scaled, X_test=X_test_scaled,
         y_train=y_train.values, y_val=y_val.values, y_test=y_test.values,
         feature_names=feature_cols)

del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

os.makedirs(f'{PROJECT_DIR}/models/bed_availability', exist_ok=True)
joblib.dump(scaler_bed, f'{PROJECT_DIR}/models/bed_availability/scaler_bed.joblib')

print(f"✓ Realistic bed data ready: {len(y_train)} train, {len(y_val)} val, {len(y_test)} test")
print(f"  Distribution: Realistic (not uniform, not biased)")
print(f"  Zero-bed cases: {(y_train == 0).sum()} in training")

In [ ]:
print("="*60)
print("DATA PREPARATION COMPLETE")
print("="*60)

files_to_check = [
    f'{processed_dir}/severity_train.npz',
    f'{processed_dir}/eta_train.npz',
    f'{processed_dir}/bed_train.npz'
]

print("\nPrepared datasets:")
for f in files_to_check:
    exists = "✓" if os.path.exists(f) else "✗"
    print(f"  {exists} {os.path.basename(f)}")

print("\n" + "="*60)
print("REALISTIC DISTRIBUTION APPLIED:")
print("="*60)
print("  ✓ 0-20 beds:  2%  (rare crisis - learnable)")
print("  ✓ 20-60 beds: 8%  (low availability)")
print("  ✓ 60-90 beds: 45% (moderate)")
print("  ✓ 90-100 beds: 55% (normal - realistic majority)")
print("\nModel learns real-world patterns, not artificial uniformity.")
print("Next: Run Notebook 03 to start training")